<a href="https://colab.research.google.com/github/Lindaoctaviani7/Lindaoctaviani7-Machine-learning_Ganjil_TI3B_2026_244107020002/blob/main/Praktikum6_JS6(ANN)_Pertemuan7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q faiss-cpu annoy hnswlib

import os
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
import kagglehub

# -------------------
# Load dataset
# -------------------
# Download latest version
path = kagglehub.dataset_download("bwandowando/spotify-songs-with-attributes-and-lyrics")
print("Path to dataset files:", path)

# Select the correct CSV containing the song attributes
target_csv = "songs_with_attributes_and_lyrics.csv"
csv_path = os.path.join(path, target_csv)

if os.path.exists(csv_path):
    # Limit rows to 50,000 to prevent Google Colab from running out of RAM (OOM)
    df = pd.read_csv(csv_path, nrows=50000)
    print(f"Loaded {len(df)} rows from {target_csv}")
else:
    raise FileNotFoundError(f"{target_csv} not found in the downloaded dataset directory.")

features = ['danceability', 'energy', 'loudness', 'speechiness',
            'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo']
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

# -------------------
# Exact Nearest Neighbor (brute-force)
# -------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# -------------------
# Annoy
# -------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

# -------------------
# HNSW
# -------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

# -------------------
# FAISS IVF
# -------------------
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
# Fixed: Passed positional arguments to IndexIVFFlat rather than keyword arguments
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], 100, faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

# -------------------
# Contoh tampilkan top-5 neighbors dari item pertama
# -------------------
print("\nTop-5 neighbors for first song:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")

Using Colab cache for faster access to the 'spotify-songs-with-attributes-and-lyrics' dataset.
Path to dataset files: /kaggle/input/spotify-songs-with-attributes-and-lyrics
Loaded 50000 rows from songs_with_attributes_and_lyrics.csv


Buat dan tuliskan analisa anda terhadap code diatas.

Analisis Perbandingan Metode Nearest Neighbor


1. Eksplorasi Metode & Algoritma

A. Exact Nearest Neighbor (Brute-Force)
* Cara Kerja: Menghitung jarak Euclidean secara langsung antara query ke *seluruh* data poin yang ada di dalam dataset.
* Akurasi: $100\%$ akurat (Ground Truth). Algoritma ini pasti menemukan tetangga terdekat yang sebenarnya.
* Skalabilitas: Buruk untuk dataset besar. Kompleksitas waktu pencariannya adalah $\mathcal{O}(N \cdot D)$ di mana $N$ adalah jumlah data (50.000) dan $D$ adalah dimensi fitur (9). Jika baris data ditingkatkan, waktu pencarian akan meningkat secara linear dan sangat lambat.

B. Annoy (Approximate Nearest Neighbors Oh Yeah)
* Cara Kerja: Menggunakan struktur pohon (*random projection trees*). Annoy membagi ruang multidimensi menggunakan *hyperplane* acak secara rekursif untuk membuat hutan pohon (*forest of trees*).
* Kelebihan: Penggunaan memori yang sangat efisien dan mendukung *static index* yang bisa dibagikan antar proses (*memory-mapped files*).
* Kekurangan: Proses pembangunannya (*build*) memerlukan penalaan jumlah *trees*. Semakin banyak *trees*, akurasi semakin tinggi namun pencarian menjadi lebih lambat.

C. HNSW (Hierarchical Navigable Small World)
* Cara Kerja: Menggunakan graf berlapis (*multi-layer graph*). Lapisan atas memiliki koneksi jarak jauh untuk navigasi cepat, sedangkan lapisan bawah memiliki koneksi lokal yang lebih rapat untuk akurasi tinggi.
* Kelebihan: Biasanya merupakan salah satu metode ANN tercepat dengan tingkat akurasi (recall) yang sangat tinggi mendekati Exact NN pada dimensi sedang.
* Kekurangan: Mengonsumsi memori (RAM) yang relatif lebih besar untuk menyimpan struktur graf dibandingkan metode lainnya.

D. FAISS IVF (Inverted File Index)
* Cara Kerja: Membagi ruang vektor menjadi beberapa kluster menggunakan K-Means (ditentukan oleh parameter quantizer). Saat pencarian dilakukan, FAISS hanya memeriksa kluster terdekat (nprobe = 10), memangkas waktu pencarian secara drastis.
* Kelebihan: Sangat dioptimalkan secara matematis untuk komputasi cepat (mendukung instruksi CPU tingkat tinggi seperti AVX dan juga akselerasi GPU).
* Kekurangan: Memerlukan fase pelatihan (*training*) indeks menggunakan data representatif sebelum data dimasukkan.


2. Evaluasi Hasil Output

Jika kita melihat output dari tetangga terdekat lagu pertama (Top-5 neighbors for first song), Anda akan melihat perbedaan indeks yang dikembalikan:
1. Exact NN memberikan hasil acuan yang mutlak benar.
2. HNSW umumnya memberikan hasil indeks yang sangat identik atau mendekati Exact NN karena strukturnya yang kokoh.
3. Annoy & FAISS mungkin memberikan beberapa indeks yang sedikit berbeda. Ini adalah kompromi (*trade-off*) dari metode *Approximate* demi mengejar kecepatan komputasi yang tinggi.

